# 🧪 LAB: Manual MLPs for Classification and Regression

In this lab, you will use `PyTorch` to implement manually a multi-layer perceptron (MLP) for three different tasks: binary classification, multi-class classification and regression. 

## General instructions to complete in ALL three tasks:

1. ***IMPLEMENTATION***:
   
Implement a separate class for each task:

  - `BinaryMLP` for the binary classification task  
  - `MultiClassMLP` for the multi-class classification task  
  - `RegressionMLP` for the regression task

   Each class must include the following methods:

  - `__init__` for initializing 1 or 2 hidden layers. (*Help*: having a second hidden layer only means the need of having a extra set of weights and intercept). 
  - `forward` to transfer information from the input to the output layer. (*Help*: make sure that information passes through a second hidden layer if there is one).
  - `cost` computing the cost.
  - `fit` for training, using the gradients to manually update the parameters of the model. **Use stochastic gradient descent to update your weights**. *N.B.* You may probably reuse much of the code we used in this week tutorial already. 
  - `predict` to convert the information at the output layer into the required output.

2. ***DATA PREPARATION***

For each task, you will be provided with a toy dataset. For each dataset:

  - Split into training and holdout sets (use an 80/20 split)
  - Standardize the features properly, avoiding data leakage
  - Convert all data into `PyTorch` Tensors for compatibility

3. ***MODEL TRAINING AND EVALUATION***

Instantiate the model for each task and train it under different hyperparameter configurations. In each case, record the performance on **both** the training and holdout sets using the appropriate metric for the task (accuracy, MSE, etc.). You should explore the following configurations:

  - One hidden layer, varying the number of hidden units (use ReLU as their activation function)
  - Same number of hidden units across one or two hidden layers (use ReLU as their activation function)
  - Repeat the above setups using Tanh activation instead of ReLU

Present your results in a compact way (e.g., a summary table, a data frame etc).

**NOTE**: When training your model, use a fixed learning rate of your choice (e.g., 0.01 is a reasonable starting point) and a reasonably large number of  epochs (e.g., 100–200) based on how training and holdout performance evolve.

4. ***REFLECTION AND DISCUSSION***

Reflect on the impact of the different hyperparameter settings:

- How does the number of hidden units affect performance?
- What changes when using two layers instead of one?
- How does the activation function (ReLU vs. Tanh) influence results?

Please elaborate your answers.

---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

**Ideally, each group member should be responsible for one of the separate tasks.** BUT, everyone should help each other along the way, both reviewing and refining results and discussion.

---

In total, this lab assignment will be worth **100 points**.

--- 
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.


In [4]:
%pip install torch scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.3/127.3 MB 17.7 MB/s  0:00:07 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 24.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 22.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 25.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 13.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
NAME(s) = "Ethan Ooi, Namitha Tholasi, Chloe Wang"

## 1. Pre-implementation Group Discussion (15 points)

Discuss and agree on:

- What cost function should be used for each of the below task.
- What changes are needed in the output layer for each of these tasks. In particular, consider the number of units and the activation function.
- Why it is important to standardize the data before training each model.
- How you could detect overfitting when training your models.

**Binary Classification (Task 1)**


**Cost function:**

We will use binary cross-entropy (log loss). It works with the probability from the sigmoid output and gives a big penalty when the model is confident but wrong.

**Output layer:**

The output layer should have 1 unit with a sigmoid activation, which gives the probability that a point is class 1. If it's 0.5 or higher, we predict class 1. Otherwise, we predict class 0.

**Why standardize the data:**

Standardizing puts both features on the same scale so one doesn't dominate the gradients. This helps gradient descent train smoothly with one learning rate. We will fit the scaler only on the training set to avoid data leakage.

**How to detect overfitting:**

We will compare training and holdout accuracy for each configuration. If training accuracy is much higher than holdout accuracy, the model is overfitting and memorizing the training data instead of learning the general pattern.

## 2. Binary Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [5]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

In [6]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#Standardize using training data only, to avoid data leakage
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

print(X_train.shape, X_test.shape)

torch.Size([800, 2]) torch.Size([200, 2])


In [ ]:
class BinaryMLP:
    def __init__(self, n_inputs, n_hidden, n_layers=1, activation="relu"):
        self.n_layers = n_layers
        self.activation = activation

        #First hidden layer
        self.W1 = torch.randn(n_inputs, n_hidden) * np.sqrt(2 / n_inputs)
        self.b1 = torch.zeros(n_hidden)
        self.params = [self.W1, self.b1]

        #Optional second hidden layer (extra set of weights and intercept)
        if n_layers == 2:
            self.W2 = torch.randn(n_hidden, n_hidden) * np.sqrt(2 / n_hidden)
            self.b2 = torch.zeros(n_hidden)
            self.params += [self.W2, self.b2]

        #Output layer: 1 unit with sigmoid
        self.W_out = torch.randn(n_hidden, 1) * np.sqrt(1 / n_hidden)
        self.b_out = torch.zeros(1)
        self.params += [self.W_out, self.b_out]

        for p in self.params:
            p.requires_grad_(True)

    def _act(self, z):
        return torch.relu(z) if self.activation == "relu" else torch.tanh(z)

    def forward(self, X):
        h = self._act(X @ self.W1 + self.b1)
        if self.n_layers == 2:
            h = self._act(h @ self.W2 + self.b2)
        return torch.sigmoid(h @ self.W_out + self.b_out)  #probability of class 1

    def cost(self, y_hat, y):
        #Binary cross-entropy
        y_hat = torch.clamp(y_hat, 1e-7, 1 - 1e-7)
        return -torch.mean(y * torch.log(y_hat) + (1 - y) * torch.log(1 - y_hat))

    def fit(self, X, y, lr=0.05, epochs=200, batch_size=16):
        n = X.shape[0]
        self.history = []
        for epoch in range(epochs):
            perm = torch.randperm(n)  #shuffle each epoch
            for i in range(0, n, batch_size):
                idx = perm[i:i + batch_size]
                loss = self.cost(self.forward(X[idx]), y[idx])
                loss.backward()
                #Manual stochastic gradient descent update
                with torch.no_grad():
                    for p in self.params:
                        p -= lr * p.grad
                        p.grad.zero_()
            with torch.no_grad():
                self.history.append(self.cost(self.forward(X), y).item())
        return self

    def predict(self, X):
        with torch.no_grad():
            return (self.forward(X) >= 0.5).float()

In [8]:
def accuracy(model, X, y):
    return (model.predict(X) == y).float().mean().item()

results = []
for act in ["relu", "tanh"]:
    for layers in [1, 2]:
        for hidden in [2, 4, 8, 16, 32]:
            torch.manual_seed(42)
            model = BinaryMLP(n_inputs=2, n_hidden=hidden, n_layers=layers, activation=act)
            model.fit(X_train, y_train, lr=0.05, epochs=200, batch_size=16)
            train_acc = accuracy(model, X_train, y_train)
            test_acc = accuracy(model, X_test, y_test)
            results.append({
                "activation": act,
                "hidden_layers": layers,
                "hidden_units": hidden,
                "train_acc": round(train_acc, 3),
                "holdout_acc": round(test_acc, 3),
                "gap": round(train_acc - test_acc, 3),
            })

results_df = pd.DataFrame(results)
results_df

,activation,hidden_layers,hidden_units,train_acc,holdout_acc,gap
0,relu,1,2,0.853,0.915,-0.062
1,relu,1,4,0.874,0.900,-0.026
2,relu,1,8,0.973,0.985,-0.012
3,relu,1,16,0.973,0.990,-0.017
4,relu,1,32,0.975,0.990,-0.015
5,relu,2,2,0.500,0.500,0.000
6,relu,2,4,0.854,0.885,-0.031
7,relu,2,8,0.978,0.990,-0.012
8,relu,2,16,0.980,0.990,-0.010
9,relu,2,32,0.978,0.990,-0.012


**Reflection for Task 1 (Binary Classification)**


**How does the number of hidden units affect performance?**

With 2 or 4 units, the model underfits and only gets about 85 to 87% accuracy because it can't fit the curved shape of the moons. At 8 units, accuracy jumps to about 97% and stays there, so adding more units doesn't help much. None of the models overfit, since holdout accuracy was always about the same as or higher than training accuracy.

**What changes when using two layers instead of one?**

For small networks, a second layer helped Tanh (86% to 96% with 2 units) but hurt ReLU, which dropped to 50% with 2 units per layer. That means it predicted the same class for every point. With 8 or more units, one and two layers gave about the same results.

**How does the activation function (ReLU vs. Tanh) influence results?**

Tanh worked better for small networks. ReLU outputs 0 for negative inputs, so some units can "die" and stop learning, which hurts a lot when there are only a few units. With 8 or more units, both reached about the same accuracy.

## 2. Multi-Class Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

---
**NOTE**: This will likely be the most challenging exercise. To help you, here are some pointers:

 - The **output (last) layer** should have as many units as there are classes in your data.  

 - The **activation function of the output layer must be softmax**, not sigmoid. Softmax ensures that all output values are between 0 and 1 and sum to 1, so they can be interpreted as probabilities across the different classes. (*Help*: you may want to look into `torch.nn.functional` to find its implementation...)

 - For a multi-class problem, your target variable `y` should be **one-hot encoded**. For example:  
   - Label = 0 → [1, 0, 0]  
   - Label = 1 → [0, 1, 0]  
   - Label = 2 → [0, 0, 1]  
   You can easily achieve this with `OneHotEncoder` from `sklearn.preprocessing`.  

 - The **predicted class** corresponds to the unit with the highest probability.  
   Example:  
   - `[0.1, 0.3, 0.6] → class 2`  
   - `[0.6, 0.2, 0.2] → class 0`  

 - For this exercise, you need to **implement the categorical cross-entropy loss** (an extension of binary cross-entropy to multiple classes). It is defined as:  

   $$\sum_{c=1}^{l} y_{o,c}\,\log(p_{o,c}),$$

   where $l$ is the number of classes, $y_{o,c}$ is the one-hot encoded label for observation $o$, and $p_{o,c}$ is the predicted probability for class $c$ (after applying softmax). The log is the natural logarithm.

---

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_classes=3, 
                           n_clusters_per_class=1, 
                           n_features=2, n_informative=2, n_redundant=0, random_state=1234, flip_y=0.15)

In [ ]:
# USE AS MANY CELLS AS NEEDED

## 3: Regression Task (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [ ]:
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=1000, n_features=2, n_informative=2, random_state=1234, noise=75)

In [ ]:
# USE AS MANY CELLS AS NEEDED

## 4. Discussion (5 points)

You created a separate class for each task and likely repeated much of the same code across implementations.

Discuss within your group how could you have leveraged inheritance to make your code more reusable and avoid duplication. Provide examples. Be specific.

YOUR TEXT HERE

## 5. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

YOUR TEXT HERE